# Módulo 50: Publicación de dashboards

Autor: Juliho Castillo Colmenares

> Lección complementaria del ejercicio de este módulo: [enunciado](m50-publicacion-de-dashboards-guia.ipynb) y [solución](m50-publicacion-de-dashboards-solucion.ipynb). No es una tarea entregada del programa.

Un tablero que solo se ejecuta en el equipo del autor no lo usa nadie más. Para que llegue a su público hay que publicarlo, mantener sus datos al día y decidir quién ve qué. Esta lección recorre ese último tramo con la app de ventas de la sección ([`../practica-ventas/app.py`](../practica-ventas/app.py)): dónde publicarla gratis, cómo preparar el repositorio, cómo manejar los secretos y la actualización de datos, cómo compartirla y cómo aplicar **seguridad por filas**, para que una misma app muestre a cada usuario solo los datos que le corresponden.

1. Opciones de publicación gratuita
2. Preparar el repositorio y las dependencias
3. Desplegar en Streamlit Community Cloud
4. Secretos
5. Actualización de los datos
6. Compartir la app
7. Seguridad por filas estática
8. Seguridad por filas dinámica
9. Límites del plan gratuito frente a autoalojar
10. Errores frecuentes
11. Ejercicio

La documentación de Streamlit citada se consultó el 18 de octubre de 2026; los servicios en la nube cambian sus condiciones con frecuencia, así que conviene revisarla antes de publicar.

In [1]:
import sys
import tomllib
from pathlib import Path

import streamlit.config
import streamlit.logger
from streamlit.testing.v1 import AppTest

# Sin mensajes del registro de Streamlit en las salidas (los errores se leen con at.exception)
streamlit.config.set_option("logger.level", "critical")
streamlit.logger.set_log_level("CRITICAL")

RAIZ = Path("../..").resolve()
PRACTICA = Path("../practica-ventas").resolve()
sys.path.insert(0, str(PRACTICA))
import estilo
import seguridad
import ventas as practica

con = practica.conectar()
APP = str(PRACTICA / "app.py")
NAFTA = ["US", "CANADA", "MEXICO"]

## 1. Opciones de publicación gratuita

Una app de Streamlit es un programa de Python que sirve una página web; publicarla es ejecutarlo en un servidor accesible desde internet. Hay tres caminos habituales:

| Opción | Cómo funciona | A favor | En contra |
| --- | --- | --- | --- |
| **Streamlit Community Cloud** | Se conecta a un repositorio de GitHub y despliega la app con un formulario | Gratis, sin servidores que mantener; se actualiza sola al hacer `git push` | Recursos limitados, apps que se duermen sin tráfico, una sola app privada |
| **Servidor propio** | `streamlit run app.py` en una máquina virtual, detrás de un proxy con HTTPS | Control total de recursos, red y acceso | Hay que instalar, asegurar y mantener el servidor |
| **Contenedor en un servicio de terceros** | Una imagen Docker con la app, en un servicio que ejecuta contenedores | Reproducible; se mueve de un proveedor a otro | Cada servicio tiene sus límites y sus planes gratuitos cambian a menudo |

Para un portafolio, Community Cloud es la opción natural: es gratuita, la mantiene el equipo de Streamlit y solo pide que el código esté en GitHub. En una herramienta de BI comercial el equivalente sería publicar en su servicio en la nube, que normalmente exige una licencia por usuario.

## 2. Preparar el repositorio y las dependencias

Community Cloud ejecuta la app **desde la raíz del repositorio**, aunque el archivo principal esté en una subcarpeta, y busca la configuración en `.streamlit/config.toml` de la raíz (documentación de *File organization*). Por eso este repositorio lanza todas las apps desde la raíz también en local:

```
certificacion-cientifico-datos/
├── .streamlit/
│   ├── config.toml              # tema (Módulo 49)
│   └── secrets.example.toml     # estructura de los secretos (sección 4)
├── pyproject.toml, uv.lock      # entorno de todo el repositorio
└── s08-dashboards/practica-ventas/
    ├── app.py                   # archivo principal
    ├── requirements.txt         # dependencias de la app
    ├── comun.py, estilo.py, seguridad.py, ventas.py
    ├── paginas/                 # inicio, resumen, medidas, analisis, acerca
    └── datos-modelo/*.parquet   # modelo que lee la app
```

**¿Qué archivo de dependencias usa?** Según la documentación de *App dependencies*, Community Cloud reconoce cinco archivos y, **en cada carpeta**, los prioriza en este orden:

| Prioridad | Archivo | Gestor |
| --- | --- | --- |
| 1 | `uv.lock` | uv |
| 2 | `Pipfile` | pipenv |
| 3 | `environment.yml` | conda |
| 4 | `requirements.txt` | pip (lo procesa con uv y recurre a pip si falla) |
| 5 | `pyproject.toml` | poetry |

Además, **busca primero en la carpeta del archivo principal y después en la raíz**, y un archivo junto al archivo principal tiene prioridad sobre uno de la raíz. La documentación recomienda usar un solo archivo de dependencias por app.

En este repositorio eso decide el diseño: en la raíz hay `uv.lock` y `pyproject.toml`, que instalarían todo el entorno de la certificación (Jupyter, scikit-learn, kaleido...). Un `requirements.txt` en `practica-ventas/`, junto a `app.py`, gana a ambos porque está en la carpeta del archivo principal, y deja la instalación en lo mínimo:

In [2]:
print((PRACTICA / "requirements.txt").read_text(encoding="utf-8"))

# Dependencias de la app para Streamlit Community Cloud (Módulo 50).
# Community Cloud busca el archivo de dependencias primero junto al archivo
# principal (esta carpeta) y después en la raíz del repositorio, así que este
# archivo tiene prioridad sobre pyproject.toml y uv.lock de la raíz.
streamlit>=1.66
plotly>=6.0
duckdb>=1.2
pandas>=2.2,<3
pyarrow>=17
# Para iniciar sesión con st.login() (proveedor OIDC), sustituye la primera línea por:
# streamlit[auth]>=1.66



Las versiones mínimas son las de un entorno ya probado (Streamlit 1.66 es la versión con la que se escribió la app) y `pandas<3` evita que una versión mayor nueva cambie el comportamiento sin aviso. Una comprobación automática confirma que cada librería externa que importa la app está en el archivo:

In [3]:
import ast

LOCALES = {"comun", "estilo", "seguridad", "ventas"}
ESTANDAR = set(sys.stdlib_module_names)


def importaciones(carpeta):
    # Módulos de primer nivel importados por los archivos .py de la app
    modulos = set()
    for archivo in [carpeta / "app.py", *carpeta.glob("*.py"), *(carpeta / "paginas").glob("*.py")]:
        for nodo in ast.walk(ast.parse(archivo.read_text(encoding="utf-8"))):
            if isinstance(nodo, ast.Import):
                modulos |= {alias.name.split(".")[0] for alias in nodo.names}
            elif isinstance(nodo, ast.ImportFrom) and nodo.level == 0:
                modulos.add(nodo.module.split(".")[0])
    return modulos - LOCALES - ESTANDAR - {"__future__"}


def requeridos(ruta):
    lineas = ruta.read_text(encoding="utf-8").splitlines()
    return {l.split("[")[0].split(">")[0].split("<")[0].split("=")[0].strip()
            for l in lineas if l.strip() and not l.startswith("#")}


externas = importaciones(PRACTICA)
lista = requeridos(PRACTICA / "requirements.txt")
print("Importa:", sorted(externas))
print("requirements.txt:", sorted(lista))
print("Faltan:", sorted(externas - lista) or "ninguna")

Importa: ['duckdb', 'pandas', 'plotly', 'streamlit']
requirements.txt: ['duckdb', 'pandas', 'plotly', 'pyarrow', 'streamlit']
Faltan: ninguna


`pyarrow` no se importa directamente, pero Streamlit lo usa para enviar tablas al navegador; se deja explícito porque es la pieza más pesada y conviene fijar su versión mínima.

**Los datos.** La app lee el modelo en Parquet de `datos-modelo/` y no los CSV originales, que el repositorio guarda con **Git LFS** (`.gitattributes`). Los archivos de LFS se guardan en el repositorio como pequeños punteros de texto, y no conviene depender de que el servidor de despliegue los descargue. Los Parquet sí son archivos normales, y pequeños:

In [4]:
import subprocess

for archivo in sorted((PRACTICA / "datos-modelo").glob("*.parquet")):
    atributos = subprocess.run(["git", "check-attr", "filter", "--", archivo.name], cwd=archivo.parent,
                               capture_output=True, text=True).stdout.strip()
    filtro = atributos.rsplit(": ", 1)[-1]
    print(f"{archivo.name:18} {archivo.stat().st_size / 1e6:6.2f} MB | filtro de git: {filtro}")
total_mb = sum(a.stat().st_size for a in (PRACTICA / "datos-modelo").glob("*.parquet")) / 1e6
print(f"Total: {total_mb:.2f} MB")

clientes.parquet     0.04 MB | filtro de git: unspecified
paises.parquet       0.00 MB | filtro de git: unspecified


productos.parquet    0.02 MB | filtro de git: unspecified


tiendas.parquet      0.01 MB | filtro de git: unspecified
ventas.parquet       8.51 MB | filtro de git: unspecified


Total: 8.57 MB


Ningún Parquet tiene el filtro `lfs` (`unspecified` significa que git los guarda tal cual), y el mayor, `ventas.parquet`, ocupa unos 8.5 MB, lejos del límite de 100 MB por archivo de GitHub. Las rutas de `ventas.py` se construyen con `Path(__file__).resolve().parent`, así que funcionan igual en Windows y en el Linux del servidor.

## 3. Desplegar en Streamlit Community Cloud

Con el repositorio en GitHub, el despliegue es un formulario (documentación de *Deploy your app*):

1. En [share.streamlit.io](https://share.streamlit.io), iniciar sesión con GitHub y elegir **Create app** → desplegar desde un repositorio existente.
2. Indicar el **repositorio**, la **rama** (por ejemplo `main`) y la **ruta del archivo principal**: `s08-dashboards/practica-ventas/app.py`.
3. Elegir, si se quiere, el subdominio de la URL (`<nombre>.streamlit.app`).
4. En **Advanced settings**, elegir la **versión de Python** y pegar los **secretos** (sección 4). Community Cloud admite las versiones de Python que siguen recibiendo actualizaciones de seguridad y usa 3.12 por defecto; este repositorio pide Python 3.11 o superior (`requires-python = ">=3.11"`), así que valen 3.11 y 3.12.
5. **Deploy**. La primera instalación tarda unos minutos; después, la app queda en su URL.

Esta lección no despliega nada: los pasos anteriores requieren cuentas propias. Todo lo que se puede comprobar sin ellas (dependencias, datos, secretos y seguridad) se comprueba en local con `AppTest`, que ejecuta la app sin servidor ni navegador:

In [5]:
at = AppTest.from_file(APP, default_timeout=90).run()
for pagina in ["paginas/resumen.py", "paginas/medidas.py", "paginas/analisis.py", "paginas/acerca.py"]:
    at.switch_page(pagina).run()
    print(f"{pagina:22} excepciones: {len(at.exception)}")

paginas/resumen.py     excepciones: 0


paginas/medidas.py     excepciones: 0


paginas/analisis.py    excepciones: 0
paginas/acerca.py      excepciones: 0


## 4. Secretos

Contraseñas, claves de API y reglas de acceso no van en el código. Streamlit los lee con `st.secrets`, que en local viene de `.streamlit/secrets.toml` (en la carpeta desde la que se lanza la app, la raíz) y en Community Cloud del campo **Secrets** de la configuración de la app, donde se pega el mismo contenido TOML. Se puede editar después del despliegue desde la configuración de la app.

El archivo real **nunca** se sube al repositorio: está en `.gitignore`. Lo que sí se sube es un ejemplo con la estructura, [`.streamlit/secrets.example.toml`](../../.streamlit/secrets.example.toml):

In [6]:
ignorados = (RAIZ / ".gitignore").read_text(encoding="utf-8").splitlines()
print("secrets.toml ignorado:", ".streamlit/secrets.toml" in ignorados)
print("secrets.toml presente:", (RAIZ / ".streamlit" / "secrets.toml").exists())

ejemplo = tomllib.loads((RAIZ / ".streamlit" / "secrets.example.toml").read_text(encoding="utf-8"))
for seccion, contenido in ejemplo.items():
    print(f"[{seccion}]", contenido)

secrets.toml ignorado: True
secrets.toml presente: False
[roles] {'NAFTA': ['US', 'CANADA', 'MEXICO'], 'SUDAMERICA': ['BRAZIL', 'COLOMBIA', 'CHILE', 'PERU'], 'EUROPA': ['SPAIN', 'PORTUGAL', 'FRANCE']}
[miembros] {'NAFTA': ['ana@empresa.com'], 'EUROPA': []}
[usuarios] {'marta@empresa.com': ['SPAIN', 'PORTUGAL', 'FRANCE'], 'jorge@empresa.com': ['US', 'MEXICO']}
[acceso] {'administradores': ['direccion@empresa.com']}


La app debe funcionar también **sin** secretos (por ejemplo, recién clonada): si no hay archivo, `st.secrets` lanza un error al leerse, así que [`seguridad.py`](../practica-ventas/seguridad.py) lo envuelve:

```python
def secretos() -> dict:
    try:
        return st.secrets.to_dict()
    except Exception:  # sin secrets.toml, st.secrets lanza un error al leerse
        return {}
```

En `AppTest`, los secretos se inyectan con `at.secrets[...]`, sin tocar ningún archivo; es lo que usan las pruebas de las secciones 7 y 8.

## 5. Actualización de los datos

En una herramienta de BI, los datos publicados se refrescan con una actualización programada. En una app de Streamlit hay tres mecanismos complementarios:

| Mecanismo | Qué hace | En la app |
| --- | --- | --- |
| **Redespliegue al hacer push** | Community Cloud vigila la rama: un `git push` actualiza la app casi al instante; si cambian las dependencias, reinstala todo | Subir un Parquet nuevo actualiza los datos |
| **Caché con caducidad** | `@st.cache_data(ttl=...)` reutiliza el resultado de una consulta durante un tiempo y luego la repite | `comun.TTL = "1h"` en todas las consultas |
| **Regenerar el modelo automáticamente** | Un flujo programado de GitHub Actions ejecuta la limpieza y sube el resultado | Ejemplo abajo (no incluido en el repositorio) |

La caché es la que hace rápida la app: la conexión a DuckDB se abre una vez por servidor con `@st.cache_resource`, y cada combinación de filtros se calcula una sola vez por hora con `@st.cache_data`. Como los filtros (incluida la lista de países permitidos) son argumentos de las funciones en caché, dos usuarios con roles distintos nunca comparten un resultado:

```python
@st.cache_data(ttl=TTL, max_entries=500, show_spinner=False)
def kpis(**filtros) -> dict:
    return ventas.kpis(conexion(), **filtros)
```

Si los datos llegaran de un sistema que cambia cada día, un flujo de GitHub Actions podría regenerar el Parquet cada mañana y subirlo; el `git push` dispara el redespliegue. Un esbozo:

```yaml
# .github/workflows/actualizar-ventas.yml (ejemplo)
name: Actualizar modelo de ventas
on:
  schedule:
    - cron: "0 6 * * *"        # todos los días a las 06:00 UTC
  workflow_dispatch:            # también a mano
jobs:
  modelo:
    runs-on: ubuntu-latest
    permissions:
      contents: write
    steps:
      - uses: actions/checkout@v4
        with:
          lfs: true             # los CSV originales están en Git LFS
      - uses: astral-sh/setup-uv@v6
      - run: uv run python s08-dashboards/practica-ventas/preparar_datos.py
      - run: |
          git config user.name "github-actions[bot]"
          git config user.email "github-actions[bot]@users.noreply.github.com"
          git add s08-dashboards/practica-ventas/datos-modelo
          git diff --cached --quiet || git commit -m "Actualizar modelo de ventas"
          git push
```

`preparar_datos.py` vuelve a ejecutar la limpieza del Módulo 44 y comprueba las cifras antes de escribir; por eso la limpieza debe ser robusta (filtrar por contenido y no por posición): se ejecutará sin nadie mirando.

## 6. Compartir la app

Según la documentación de *Share your app*, en Community Cloud:

- La **visibilidad sigue al repositorio**: un repositorio público produce una app pública, y uno privado, una app privada. Solo los desarrolladores pueden cambiarla.
- Una app privada solo la ven los desarrolladores y las personas **invitadas por correo** (botón **Share** o *App settings → Sharing*). Los invitados entran con su cuenta de Google o con un enlace de un solo uso enviado al correo.
- El plan gratuito permite **una sola app privada** a la vez.
- Una app sin tráfico durante 12 horas **se duerme**; cualquiera con acceso la despierta al visitarla, tras una breve espera.

La lista de invitados controla **quién entra**, pero no **qué ve** cada uno: todos los invitados ven la misma app con los mismos datos. Para eso hace falta la seguridad por filas.

## 7. Seguridad por filas estática

La **seguridad por filas** (*row-level security*, RLS) filtra las filas que puede ver cada usuario. Una sola app sirve a todos: el gerente de Norteamérica ve sus tres países y el de Europa los suyos, sin una app por región.

En la versión **estática**, cada **rol** tiene una lista fija de países, definida en los secretos:

```toml
[roles]
NAFTA = ["US", "CANADA", "MEXICO"]
SUDAMERICA = ["BRAZIL", "COLOMBIA", "CHILE", "PERU"]
EUROPA = ["SPAIN", "PORTUGAL", "FRANCE"]

[miembros]               # qué correos pertenecen a cada rol
NAFTA = ["ana@empresa.com"]
```

Lo que en una herramienta de BI sería una regla DAX sobre la tabla de países (`[Country] IN {"US", "CANADA", "MEXICO"}`) aquí es el argumento `paises` de las consultas. La clave del diseño es **dónde** se aplica el filtro: dentro de `comun.filtros()`, que todas las páginas usan antes de consultar, de modo que la lista llega a DuckDB como parte del `WHERE` y ninguna página llega a tener filas de otros países. Las reglas se resuelven con funciones puras, que se prueban sin Streamlit:

In [7]:
reglas = ejemplo  # las del archivo de ejemplo
for opcion in ["todos", "rol:NAFTA", "rol:EUROPA", "rol:INVENTADO"]:
    print(f"{opcion:14} -> {seguridad.paises_de_opcion(opcion, reglas)}")

ventas_nafta = practica.kpis(con, paises=NAFTA)["ventas_total"]
print("Ventas del rol NAFTA:", estilo.entero(ventas_nafta), f"({ventas_nafta / practica.VENTAS_TOTALES:.2%} del total)")

todos          -> None
rol:NAFTA      -> ['US', 'CANADA', 'MEXICO']
rol:EUROPA     -> ['SPAIN', 'PORTUGAL', 'FRANCE']
rol:INVENTADO  -> []
Ventas del rol NAFTA: 16 364 381 054 (39.02% del total)


`None` significa "sin restricción" y una lista vacía, "ninguna fila": un rol inexistente no ve nada en lugar de verlo todo. El rol NAFTA debe ver 16 364 381 054, el 39.02 % de las ventas.

**Probar el rol.** En una herramienta de BI se usa "Ver como". La app tiene su equivalente: si los secretos **no** tienen sección `[auth]` (no hay inicio de sesión), la barra lateral muestra un selector "Ver como (solo demo)" con los roles y usuarios. Con `AppTest` y `at.secrets` se comprueba que elegir NAFTA deja solo esos tres países y que el total coincide con `ventas.kpis`:

In [8]:
at = AppTest.from_file(APP, default_timeout=90)
at.secrets["roles"] = {"NAFTA": NAFTA, "EUROPA": ["SPAIN", "PORTUGAL", "FRANCE"]}
at.run()
print("Opciones de 'Ver como':", at.selectbox(key="ver_como").options)

at.selectbox(key="ver_como").select("rol:NAFTA").run()
at.switch_page("paginas/resumen.py").run()
tabla = at.dataframe[0].value
print("Países permitidos:", at.session_state["paises_permitidos"])
print("Países en la tabla del resumen:", sorted(tabla["pais"]))
print("Tarjeta de ventas:", at.metric[0].value, "| esperado:", estilo.millones(ventas_nafta))
assert sorted(tabla["pais"]) == sorted(NAFTA)
assert at.metric[0].value == estilo.millones(ventas_nafta)

Opciones de 'Ver como': ['Todos los países (sin restricción)', 'Rol NAFTA', 'Rol EUROPA']


Países permitidos: ['US', 'CANADA', 'MEXICO']
Países en la tabla del resumen: ['CANADA', 'MEXICO', 'US']
Tarjeta de ventas: 16 364.38 M | esperado: 16 364.38 M


El filtro de países de la barra lateral también se recorta a los permitidos: con el rol NAFTA, el selector "Países" solo ofrece esos tres, y si un usuario manipulara la petición para pedir otro país, `filtros()` lo descartaría al intersectar con la lista permitida.

Este selector **no es seguridad**: cualquiera puede elegir "Todos los países". Sirve para probar las reglas, como "Ver como" en un programa de escritorio. En producción la identidad debe venir de un inicio de sesión real.

## 8. Seguridad por filas dinámica

Con muchos usuarios, mantener un rol por región se vuelve pesado. La versión **dinámica** usa una tabla que asigna países a cada **correo**, y la regla depende de quién está mirando. En una herramienta de BI la función `USERPRINCIPALNAME()` devuelve el correo del usuario conectado; en Streamlit, ese papel lo cumple `st.user.email`, que existe después de un inicio de sesión con `st.login()`.

```toml
[usuarios]
"marta@empresa.com" = ["SPAIN", "PORTUGAL", "FRANCE"]
"jorge@empresa.com" = ["US", "MEXICO"]

[acceso]
administradores = ["direccion@empresa.com"]   # ven todo
```

`seguridad.paises_de_correo` une los países de los roles a los que pertenece el correo (`[miembros]`) y los de su fila en `[usuarios]`; los administradores ven todo y un correo desconocido no ve nada:

In [9]:
for correo in ["jorge@empresa.com", "MARTA@empresa.com", "ana@empresa.com", "direccion@empresa.com",
               "intruso@otra.com", None]:
    print(f"{str(correo):24} -> {seguridad.paises_de_correo(correo, ejemplo)}")

ventas_jorge = practica.kpis(con, paises=["US", "MEXICO"])["ventas_total"]
print("Ventas que debe ver jorge:", estilo.entero(ventas_jorge))
print("Ventas con una lista vacía:", practica.kpis(con, paises=[])["transacciones"], "transacciones")

jorge@empresa.com        -> ['MEXICO', 'US']
MARTA@empresa.com        -> ['FRANCE', 'PORTUGAL', 'SPAIN']
ana@empresa.com          -> ['CANADA', 'MEXICO', 'US']
direccion@empresa.com    -> None
intruso@otra.com         -> []
None                     -> []
Ventas que debe ver jorge: 9 680 856 963
Ventas con una lista vacía: 0 transacciones


Para dar acceso a una persona nueva basta con añadir una línea a `[usuarios]` en los secretos: no hay que tocar el código. El selector de demostración también ofrece los usuarios, lo que equivale a "Ver como otro usuario":

In [10]:
at = AppTest.from_file(APP, default_timeout=90)
at.secrets["usuarios"] = {"jorge@empresa.com": ["US", "MEXICO"]}
at.run()
at.selectbox(key="ver_como").select("usuario:jorge@empresa.com").run()
at.switch_page("paginas/resumen.py").run()
print("Países:", sorted(at.dataframe[0].value["pais"]), "| ventas:", at.metric[0].value,
      "| esperado:", estilo.millones(ventas_jorge))
assert at.metric[0].value == estilo.millones(ventas_jorge)

Países: ['MEXICO', 'US'] | ventas: 9 680.86 M | esperado: 9 680.86 M


**La vía de producción: `st.login()`.** Para que `st.user.email` sea fiable, el correo tiene que venir de un proveedor de identidad (Google, Microsoft Entra ID, Auth0, Okta...) mediante OpenID Connect (OIDC), nunca de un cuadro de texto que cualquiera puede rellenar. Se configura con una sección `[auth]` en los secretos (la plantilla está comentada en `secrets.example.toml`) y requiere instalar `streamlit[auth]`:

```toml
[auth]
redirect_uri = "https://<tu-app>.streamlit.app/oauth2callback"
cookie_secret = "<una cadena larga y aleatoria>"
client_id = "<del registro de la app en el proveedor>"
client_secret = "<del registro de la app en el proveedor>"
server_metadata_url = "https://accounts.google.com/.well-known/openid-configuration"
```

Con esa sección, `seguridad.py` cambia de modo: desaparece el selector de demostración, la barra lateral muestra un botón que llama a `st.login()` y los países salen de `paises_de_correo(st.user.email, ...)`. El proveedor firma la identidad y Streamlit la guarda en una cookie firmada con `cookie_secret`, así que el usuario no puede hacerse pasar por otro.

**Esta parte no se puede probar aquí:** hace falta registrar la app en un proveedor OIDC y obtener credenciales reales. Lo que sí se comprueba es el comportamiento sin sesión: con `[auth]` configurado y nadie conectado, la app no muestra datos.

In [11]:
prod = AppTest.from_file(APP, default_timeout=90)
prod.secrets["auth"] = {
    "redirect_uri": "http://localhost:8501/oauth2callback", "cookie_secret": "solo-para-la-prueba",
    "client_id": "x", "client_secret": "x",
    "server_metadata_url": "https://accounts.google.com/.well-known/openid-configuration",
}
prod.run()
print("Selector 'Ver como':", "ver_como" in [s.key for s in prod.selectbox])
print("Botones de la barra lateral:", [b.label for b in prod.sidebar.button][:2])
prod.switch_page("paginas/resumen.py").run()
print("Países permitidos sin sesión:", prod.session_state["paises_permitidos"])
print("Mensaje:", prod.info[0].value, "| tarjetas:", len(prod.metric))

Selector 'Ver como': False
Botones de la barra lateral: ['Iniciar sesión', '2019, primer semestre']
Países permitidos sin sesión: []
Mensaje: Inicia sesión desde la barra lateral para ver los datos. | tarjetas: 0


Sin sesión, la lista de países es vacía, la página se detiene antes de consultar y no aparece ninguna tarjeta. Es el equivalente de "los visores sin rol asignado no ven datos".

## 9. Límites del plan gratuito frente a autoalojar

La documentación de *Manage your app* da estos límites aproximados (con fecha de febrero de 2024, y avisa de que pueden cambiar sin previo aviso):

| Recurso | Community Cloud |
| --- | --- |
| CPU | de 0.078 a 2 núcleos |
| Memoria | de 690 MB a 2.7 GB |
| Almacenamiento | hasta 50 GB |
| Inactividad | la app se duerme tras 12 horas sin tráfico |
| Apps privadas | una a la vez |

La app de ventas cabe con holgura: el modelo pesa unos 8.5 MB en Parquet, DuckDB lo consulta sin cargarlo entero en memoria y a la app solo llegan tablas agregadas. Cuándo conviene autoalojar:

| Necesidad | Community Cloud | Autoalojar (servidor o contenedor) |
| --- | --- | --- |
| Coste | Gratis | Pagar la máquina o el servicio |
| Mantenimiento | Ninguno | Actualizaciones, HTTPS, copias, monitoreo |
| Datos que no pueden salir de la empresa | No | Sí, en la red propia |
| Muchas apps privadas o muchos usuarios a la vez | Limitado | Lo que dé la máquina |
| Siempre despierta | No | Sí |
| Inicio de sesión | `st.login()` con OIDC | `st.login()` con OIDC, o el proxy de la empresa |

## 10. Errores frecuentes

**Subir `secrets.toml` al repositorio.** Una vez en el historial de git, el secreto está expuesto aunque se borre después, y hay que cambiarlo. `.gitignore` lo evita; antes de cada commit conviene revisar `git status`.

**Olvidar una dependencia.** En local todo funciona porque el entorno de la certificación tiene de todo; en el servidor, `requirements.txt` es lo único que se instala y la app falla con `ModuleNotFoundError`. La comprobación de la sección 2 lo detecta antes de desplegar:

In [12]:
sin_plotly = lista - {"plotly"}
print("Si faltara plotly en requirements.txt, faltarían:", sorted(externas - sin_plotly))

Si faltara plotly en requirements.txt, faltarían: ['plotly']


**Depender de Git LFS para los datos.** Si el servidor no descarga los archivos de LFS, lo que encuentra en lugar del Parquet es un puntero de texto, y la lectura falla:

In [13]:
import tempfile

import duckdb

puntero = Path(tempfile.mkdtemp()) / "ventas.parquet"
puntero.write_text("version https://git-lfs.github.com/spec/v1\n"
                   "oid sha256:0000000000000000000000000000000000000000000000000000000000000000\n"
                   "size 8507105\n", encoding="utf-8")
try:
    duckdb.sql(f"SELECT count(*) FROM read_parquet('{puntero.as_posix()}')").fetchone()
except duckdb.Error as error:
    print(type(error).__name__, "->", str(error).split(" at end of file")[0])

InvalidInputException -> Invalid Input Error: No magic bytes found


**Filtrar solo al dibujar.** Aplicar la seguridad después de consultar (por ejemplo, filtrar la tabla ya calculada) parece equivalente, pero las medidas se calcularon con todas las filas. En la tabla por país, el porcentaje de unidades revela el peso de los países que el usuario no debería ver:

In [14]:
mal = practica.tabla_por_pais(con)
mal = mal[mal["pais"].isin(NAFTA)]
bien = practica.tabla_por_pais(con, paises=NAFTA)
print(f"Filtrando después: los tres países suman {mal['pct_cantidad'].sum():.1%} de las unidades")
print(f"Filtrando en la consulta: suman {bien['pct_cantidad'].sum():.1%}")

Filtrando después: los tres países suman 30.0% de las unidades
Filtrando en la consulta: suman 100.0%


Con el filtro aplicado después, el usuario de NAFTA deduce que el resto del mundo vende el 70 % de las unidades. Por eso el filtro va en el argumento `paises`, antes de cualquier cálculo, y también la portada de la app calcula su resumen solo con los países permitidos.

**Confiar en el valor que manda el navegador.** Las opciones de un `st.selectbox` se validan en el navegador, no en el servidor; un cliente modificado puede enviar un valor que no estaba en la lista. `paises_permitidos()` comprueba que la opción existe y, si no, usa un valor seguro; `paises_de_opcion` devuelve una lista vacía para un rol desconocido.

**Tomar el correo de un cuadro de texto.** `st.text_input("Tu correo")` no es autenticación: cualquiera puede escribir el correo de otro. El correo solo es fiable si viene de `st.user` después de `st.login()`.

**Confundir `None` con una lista vacía.** En `ventas.py`, `paises=None` no filtra y `paises=[]` no deja ninguna fila. Si la función de seguridad devolviera `None` para un usuario sin asignación, ese usuario lo vería todo. La regla es "denegar por defecto": solo los administradores reciben `None`.

## 11. Ejercicio

Una empresa quiere distribuir la app de ventas a sus gerentes regionales.

1. Escribe la sección de secretos de un rol `EUROPA` que vea solo `SPAIN`, `PORTUGAL` y `FRANCE`, y comprueba con `seguridad.paises_de_opcion` que la regla devuelve esos países.
2. Calcula las ventas totales que debería ver el rol a partir de las ventas por país, y compruébalo con `AppTest` eligiendo el rol en "Ver como".
3. La empresa prefiere la seguridad dinámica. Escribe la sección `[usuarios]` para que `marta@empresa.com` vea Europa y `jorge@empresa.com` vea `US` y `MEXICO`, y calcula las ventas de cada uno.
4. Indica qué pasos hacen falta para que Marta vea la app publicada con su restricción.
5. Los archivos de ventas originales cambian cada noche. ¿Qué hace falta para que la app muestre los datos nuevos cada mañana?

### Solución

In [15]:
toml_europa = '''
[roles]
EUROPA = ["SPAIN", "PORTUGAL", "FRANCE"]
'''
reglas_europa = tomllib.loads(toml_europa)
europa = seguridad.paises_de_opcion("rol:EUROPA", reglas_europa)
print("Rol EUROPA:", europa)

por_pais = practica.ventas_por(con, "pais").set_index("pais")["ventas_total"]
print(por_pais[europa].map(estilo.entero).to_string())
ventas_europa = int(por_pais[europa].sum())
print("Total:", estilo.entero(ventas_europa), f"({ventas_europa / practica.VENTAS_TOTALES:.2%} del total)")

at = AppTest.from_file(APP, default_timeout=90)
at.secrets["roles"] = reglas_europa["roles"]
at.run()
at.selectbox(key="ver_como").select("rol:EUROPA").run()
at.switch_page("paginas/resumen.py").run()
print("Ver como EUROPA:", at.metric[0].value, "| países:", sorted(at.dataframe[0].value["pais"]))
assert at.metric[0].value == estilo.millones(ventas_europa)

Rol EUROPA: ['SPAIN', 'PORTUGAL', 'FRANCE']
pais
SPAIN       2 360 722 916
PORTUGAL    2 401 536 230
FRANCE      6 709 452 115
Total: 11 471 711 261 (27.35% del total)


Ver como EUROPA: 11 471.71 M | países: ['FRANCE', 'PORTUGAL', 'SPAIN']


In [16]:
toml_usuarios = '''
[usuarios]
"marta@empresa.com" = ["SPAIN", "PORTUGAL", "FRANCE"]
"jorge@empresa.com" = ["US", "MEXICO"]
'''
reglas_usuarios = tomllib.loads(toml_usuarios)
for correo in reglas_usuarios["usuarios"]:
    paises = seguridad.paises_de_correo(correo, reglas_usuarios)
    print(f"{correo:20} {paises} -> {estilo.entero(practica.kpis(con, paises=paises)['ventas_total'])}")

marta@empresa.com    ['FRANCE', 'PORTUGAL', 'SPAIN'] -> 11 471 711 261
jorge@empresa.com    ['MEXICO', 'US'] -> 9 680 856 963


1. El rol `EUROPA` se define en `[roles]` y la regla devuelve `['SPAIN', 'PORTUGAL', 'FRANCE']`.
2. Ventas del rol = 2 360 722 916 + 2 401 536 230 + 6 709 452 115 = **11 471 711 261**, el 27.35 % del total. "Ver como EUROPA" muestra 11 471.71 M y solo esos tres países.
3. Con la tabla `[usuarios]`, Marta ve los mismos 11 471 711 261 que el rol EUROPA, y Jorge ve 5 025 305 951 + 4 655 551 012 = 9 680 856 963.
4. Para Marta: añadir su línea a `[usuarios]` en los secretos de la app publicada (*Settings → Secrets*); configurar la sección `[auth]` con un proveedor OIDC (y `streamlit[auth]` en `requirements.txt`), registrando como `redirect_uri` la URL de la app; y darle acceso: si la app es privada, invitarla por correo desde **Share**. Al entrar, pulsa "Iniciar sesión", el proveedor confirma su correo y la app filtra a sus tres países. Si su correo no estuviera en `[usuarios]` ni en ningún rol, no vería datos.
5. Si los archivos originales llegan a un lugar accesible desde GitHub, un flujo programado de GitHub Actions (sección 5) puede ejecutar `preparar_datos.py` cada mañana y subir el Parquet nuevo; el `git push` redespliega la app. La caché con `ttl="1h"` garantiza que, como mucho una hora después, todas las sesiones lean los datos nuevos aunque el servidor no se reinicie.

## Resumen

- Streamlit Community Cloud publica gratis una app desde un repositorio de GitHub: repositorio, rama, archivo principal, versión de Python y secretos. La app se ejecuta desde la raíz del repositorio y lee `.streamlit/config.toml` de allí.
- Community Cloud busca el archivo de dependencias primero junto al archivo principal y luego en la raíz, con la prioridad `uv.lock` > `Pipfile` > `environment.yml` > `requirements.txt` > `pyproject.toml`; un `requirements.txt` mínimo junto a `app.py` evita instalar todo el entorno del repositorio.
- Los secretos se leen con `st.secrets`, viven en `.streamlit/secrets.toml` (ignorado por git) o en la configuración de la app, y se documentan con un archivo de ejemplo.
- Los datos se actualizan con un `git push` (redespliegue), con `st.cache_data(ttl=...)` y, si hace falta, con un flujo programado de GitHub Actions.
- La seguridad por filas se aplica en las consultas, con el argumento `paises`: estática por rol, o dinámica por correo con `st.user.email` después de `st.login()`. Se deniega por defecto y se prueba con "Ver como" y `AppTest`.
- El plan gratuito tiene recursos limitados y una sola app privada; autoalojar da control a cambio de mantenimiento.

Con este módulo termina la sección de dashboards. En el **Módulo 51** comienza la sección de modelado, con el [análisis de regresión](../../s09-modelado/m51-analisis-de-regresion/m51-analisis-de-regresion-leccion.ipynb).